# Week 3 Day 2 — Hugging Face Pipelines

**Local Mac reference notebook.** This is the Day 2 lab adapted off Colab: same pipeline tasks, but device selection uses Apple **MPS** (or CUDA/CPU), and the Hugging Face token comes from `.env` instead of Colab secrets.

## What this notebook is for

Hugging Face exposes two API levels:

| Level | What it is | When to use it |
| --- | --- | --- |
| **High-level: `pipeline()`** | One function that loads a model, tokenizer, and post-processing for a named task | Everyday inference: classify, translate, summarize, generate |
| **Low-level: models + tokenizers** | You load `AutoModel` / `AutoTokenizer` yourself | Fine-tuning, custom heads, training (later weeks) |

This notebook is only about **inference** with `pipeline()`.

**Training** updates a model's weights on new data (or fine-tunes an already-trained model). **Inference** runs a model that is already trained. GPT/Claude/Gemini API calls from Weeks 1–2 were inference. Pipelines are the same idea for open-source models you run locally.

## How to use this as a cheat sheet

1. Run the **Setup** cells once per kernel (imports, `device`, Hugging Face login).
2. Jump to the task you need. Each section is independent except for Setup.
3. Pattern to copy: `pipe = pipeline(task, model=..., device=device)` then `pipe(input)`.
4. If you do not pass `model=`, Hugging Face picks a default. Fine for learning; **name the model in production**.

Course Colab (original): [Week 3 Day 2 pipelines](https://colab.research.google.com/drive/1aMaEw8A56xs0bRM4lu8z7ou18jqyybGm?usp=sharing)


## Setup

Packages live in this project's `.venv`. This environment has no `pip` module, so skip `%pip` / `!pip` here. `datasets` and `transformers` were installed in Day 1.

Local differences from the Colab notebook:

- No `google.colab.userdata` — load `HF_TOKEN` from `.env`
- No `nvidia-smi` / `device="cuda"` — use MPS on this Mac
- `pipeline(..., device=device)` where `device` is a `torch.device`


In [ ]:
# Confirm the packages this notebook needs. Re-run Setup after a kernel restart.
import datasets
import diffusers
import torch
import transformers

print(f"torch         {torch.__version__}")
print(f"transformers  {transformers.__version__}")
print(f"datasets      {datasets.__version__}")
print(f"diffusers     {diffusers.__version__}")


In [ ]:
import os

import torch
from datasets import load_dataset
from diffusers import AutoPipelineForText2Image
from dotenv import load_dotenv
from huggingface_hub import login
from IPython.display import Audio, display
from transformers import pipeline


### Device

Pipelines run **inference**, not training. Putting the model on a GPU (MPS on Apple Silicon, CUDA on NVIDIA) makes that inference faster.

`pipeline()` accepts a `torch.device`, a string like `"mps"` / `"cuda"`, or an index (`0` = first GPU, `-1` = CPU).


In [ ]:
# Prefer Apple GPU, then NVIDIA, then CPU. Reuse this `device` in every pipeline below.
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

# float16 is the usual inference dtype on GPU; CPU stays float32.
dtype = torch.float16 if device.type in ("mps", "cuda") else torch.float32

print(f"Device: {device}")
print(f"dtype:  {dtype}")


### Hugging Face login

Needed for gated models and to avoid anonymous rate limits. Create a token at https://huggingface.co/settings/tokens (write scope if you will push models later) and put `HF_TOKEN=hf_...` in the project `.env`.


In [ ]:
load_dotenv(override=True)
hf_token = os.getenv("HF_TOKEN")

# Tokens always start with hf_. login() returns None — do not assign it to hf_token.
if hf_token and hf_token.startswith("hf_"):
    print("HF key looks good so far")
else:
    raise ValueError("HF_TOKEN not found. Add HF_TOKEN=hf_... to your .env file.")

login(hf_token, add_to_git_credential=True)


## How `pipeline()` works

```python
my_pipeline = pipeline(task, model=optional_id, device=device)
result = my_pipeline(my_input)
```

- **task** — a string such as `"sentiment-analysis"`, `"ner"`, `"summarization"`. See the catalog at the end.
- **model** — a Hub id. Omit it and Hugging Face chooses a default (you will see a warning).
- **device** — where tensors run. On this Mac that is `mps`.

Create the pipeline once, then call it on as many inputs as you want. Loading weights is the slow part.


## Sentiment analysis

**What it does:** classifies text as positive/negative (or star ratings, depending on the model).

**Default model:** `distilbert/distilbert-base-uncased-finetuned-sst-2-english` — binary SST-2 labels.

**Richer model:** `nlptown/bert-base-multilingual-uncased-sentiment` — 1–5 stars, better at mixed or hedged wording.

**Returns:** `[{'label': ..., 'score': ...}]`. `score` is the model's confidence in that label, not a general "how strong is the feeling" meter.


In [ ]:
# Default binary classifier. First call downloads weights; later calls reuse this object.
my_simple_sentiment_analyzer = pipeline("sentiment-analysis", device=device)

result = my_simple_sentiment_analyzer(
    "I'm super excited to be on the way to LLM mastery!"
)
print(result)


In [ ]:
# Hedged wording. The binary model still leans POSITIVE — it only has two buckets.
result = my_simple_sentiment_analyzer(
    "I should be more excited to be on the way to LLM mastery!"
)
print(result)


In [ ]:
# Same sentence, 5-star model. Expect a middle rating (around 3 stars) instead of POSITIVE.
better_sentiment = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    device=device,
)
result = better_sentiment(
    "I should be more excited to be on the way to LLM mastery!!"
)
print(result)


## Named entity recognition (NER)

**What it does:** finds spans in text and labels them (person, organization, location, miscellaneous).

**Default model:** `dbmdz/bert-large-cased-finetuned-conll03-english` (CoNLL-03 tags). A warning about unused `pooler` weights is expected — that layer is not used for token classification.

**Returns:** one dict per **token**, not per entity. WordPiece models split names (`HuggingFace` → `Hu`, `##gging`, `##F`, `##ace`). Use `aggregation_strategy="simple"` when you want merged spans.

Tags: `PER` person, `ORG` organization, `LOC` location, `MISC` other. Prefix `B-` = beginning of a span, `I-` = inside a span.


In [ ]:
ner = pipeline("ner", device=device)
text = (
    "AI Engineers are learning about the amazing pipelines from "
    "HuggingFace in Google Colab from Ed Donner"
)

print("Token-level (default):")
for entity in ner(text):
    print(entity)

# Merge WordPieces into whole entities for anything you would show a user.
ner_merged = pipeline("ner", device=device, aggregation_strategy="simple")
print("\nMerged entities:")
for entity in ner_merged(text):
    print(entity)


## Question answering with context

**What it does:** extracts an answer **from the context you pass**. It does not search the web or use model knowledge outside that string.

**Default model:** `distilbert/distilbert-base-cased-distilled-squad` (SQuAD extractive QA).

**Call shape:** `pipe(question=..., context=...)` — keyword arguments, not a single string.

**Returns:** `{'score', 'start', 'end', 'answer'}`. `start` / `end` are character offsets into `context`.


In [ ]:
question = "What are Hugging Face pipelines?"
# Context must actually contain the answer. This is extractive QA, not open-ended chat.
context = "Pipelines are a high level API for inference of LLMs with common tasks"

question_answerer = pipeline("question-answering", device=device)
result = question_answerer(question=question, context=context)
print(result)


## Text summarization

**What it does:** compresses a passage. The default is an encoder-decoder (BART family), not an LLM chat model.

**Default model:** `sshleifer/distilbart-cnn-12-6`.

**Useful kwargs:**

- `max_length` / `min_length` — token bounds for the summary
- `do_sample=False` — greedy decoding, so the summary is deterministic


In [ ]:
summarizer = pipeline("summarization", device=device)

text = """
The Hugging Face transformers library is an incredibly versatile
and powerful tool for natural language processing (NLP). It allows
users to perform a wide range of tasks such as text classification,
named entity recognition, and question answering, among others. It's
an extremely popular library that's widely used by the open-source
data science community. It lowers the barrier to entry into the
field by providing Data Scientists with a productive, convenient
way to work with transformer models.
"""

# do_sample=False → same summary every run. Raise max_length if the output looks cut off.
summary = summarizer(text, max_length=50, min_length=25, do_sample=False)
print(summary[0]["summary_text"])


## Translation

**What it does:** translates between a language pair. Task names like `translation_en_to_fr` select a default model for that pair. For any other pair, pass `model=` (Helsinki-NLP `opus-mt-*` checkpoints are the usual starting point).

**Returns:** `[{'translation_text': ...}]`. Domain words can come out literal — `pipeline` often becomes Spanish *tuberías* (water pipes) rather than the ML sense.


In [ ]:
# Task name picks the default English → French model.
translator_fr = pipeline("translation_en_to_fr", device=device)
result = translator_fr(
    "The Data Scientists were truly amazed by the power and simplicity "
    "of the HuggingFace pipeline API."
)
print(result[0]["translation_text"])


In [ ]:
# Explicit model for English → Spanish. Browse more at:
# https://huggingface.co/models?pipeline_tag=translation
translator_es = pipeline(
    "translation_en_to_es",
    model="Helsinki-NLP/opus-mt-en-es",
    device=device,
)
result = translator_es(
    "The Data Scientists were truly amazed by the power and simplicity "
    "of the HuggingFace pipeline API."
)
print(result[0]["translation_text"])


## Zero-shot classification

**What it does:** scores text against **labels you provide at call time**. No extra fine-tuning. Internally this is a natural language inference (NLI) model: it asks whether the text entails each label.

**Returns:** `labels` sorted by `scores`.


In [ ]:
classifier = pipeline("zero-shot-classification", device=device)
result = classifier(
    "Hugging Face's Transformers library is amazing!",
    candidate_labels=["technology", "sports", "politics"],
)
print(result)


## Text generation

**What it does:** continues a prompt. The default is a small GPT-2-style model, so quality is much lower than hosted GPT/Claude. Useful to see the pipeline shape, not as a production LLM.

**Tip:** set `max_new_tokens` so generation cannot run away. Sampling (`do_sample=True`) makes output vary between runs.


In [ ]:
generator = pipeline("text-generation", device=device)
result = generator(
    "If there's one thing I want you to remember about using HuggingFace pipelines, it's",
    max_new_tokens=40,
    truncation=True,
)
print(result[0]["generated_text"])


## Image generation (Diffusers)

Transformers `pipeline()` covers NLP/audio/vision-understanding tasks. **Image generation** uses the sibling library **Diffusers**, with a similar pattern: load a pipeline, `.to(device)`, call it with a prompt.

`stabilityai/sdxl-turbo` is a distilled SDXL checkpoint: few steps (`num_inference_steps=4`) and `guidance_scale=0.0` are the recommended turbo settings, not a general Stable Diffusion recipe.

This cell is slower and uses more memory than the NLP examples. Skip it if you only need the text pipelines.


In [ ]:
load_kwargs = {"torch_dtype": dtype}
if dtype == torch.float16:
    load_kwargs["variant"] = "fp16"

pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo",
    **load_kwargs,
)
pipe.to(device)

prompt = "A class of students learning AI engineering in a vibrant pop-art style"
# Turbo recipe: 4 steps, guidance 0. Default SDXL would use ~20–50 steps and guidance ~7.
image = pipe(prompt=prompt, num_inference_steps=4, guidance_scale=0.0).images[0]
display(image)


## Audio generation (text-to-speech)

**Model:** `microsoft/speecht5_tts`. It needs a **speaker embedding** (a voice fingerprint) from `Matthijs/cmu-arctic-xvectors`. Index `7306` is the embedding the course uses.

**Device:** SpeechT5 is unreliable on Apple MPS and can crash the kernel. Run it on **CPU** here (or CUDA if you have it). Optional: skip this cell when iterating on the NLP sections.


In [ ]:
# SpeechT5 is unstable on MPS; keep it on CUDA if present, otherwise CPU.
tts_device = device if device.type == "cuda" else torch.device("cpu")

synthesiser = pipeline("text-to-speech", "microsoft/speecht5_tts", device=tts_device)

embeddings_dataset = load_dataset(
    "Matthijs/cmu-arctic-xvectors",
    split="validation",
    trust_remote_code=True,
)
# unsqueeze(0) adds a batch dimension: (embedding_size,) → (1, embedding_size).
speaker_embedding = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0)

speech = synthesiser(
    "Hi to an artificial intelligence engineer, on the way to mastery!",
    forward_params={"speaker_embeddings": speaker_embedding},
)
Audio(speech["audio"], rate=speech["sampling_rate"])


## Pipeline catalog

**Transformers tasks** (expand *task* on this page):  
https://huggingface.co/docs/transformers/main_classes/pipelines

**Diffusers pipelines:**  
https://huggingface.co/docs/diffusers/en/api/pipelines/overview

Common Transformers task strings used in this notebook:

| Task string | Typical use |
| --- | --- |
| `sentiment-analysis` / `text-classification` | Label a sentence |
| `ner` / `token-classification` | Find entities in text |
| `question-answering` | Extract an answer from context |
| `summarization` | Compress a passage |
| `translation` / `translation_en_to_fr` | Language pairs |
| `zero-shot-classification` | Labels chosen at call time |
| `text-generation` | Continue a prompt |
| `text-to-speech` | Audio from text |


## Conclusion

`pipeline(task)` is the fastest way to run a trained Hugging Face model: it wires up the tokenizer, model, device, and output decoding. That is the whole Day 2 idea.

**Takeaways to reuse later**

1. **Create once, call many times.** Loading weights dominates runtime.
2. **Pass `device` explicitly.** On this machine that is MPS; Colab uses `"cuda"`.
3. **Name the model in anything you keep.** Defaults change and the Hub warning is there for a reason.
4. **Read the return type.** Sentiment is a list of dicts; QA is one dict; summarization/translation use `result[0]["..._text"]`; NER is a list of token (or aggregated) spans.
5. **QA is extractive.** The answer has to appear in `context`.
6. **Diffusers is the image-generation cousin** of Transformers pipelines, not a different product philosophy.
7. **Not every checkpoint likes MPS.** NLP pipelines in this notebook are fine on MPS; SpeechT5 should stay on CPU.

**When pipelines are the wrong tool**

- Fine-tuning or training (later weeks: Trainer / low-level `AutoModel`)
- Custom preprocessing, logits, or multi-step graphs
- Hosted large LLMs (OpenAI / Anthropic / Gemini APIs, as in Weeks 1–2)

**Local Mac checklist if something breaks**

- Kernel restart → re-run Setup (imports, `device`, `login`)
- `userdata` / `device="cuda"` / `!pip` → Colab-only; this notebook already avoids them
- Gated Hub model → confirm `HF_TOKEN` and that the account accepted the model license
